# Week 33, Day 3: Facilitation and the synapse as a filter

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w033/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The full model

Complete `tm(times, U, tauD, tauF)`, which returns u x at each spike time (ms). Between spikes, x recovers towards 1 with τ_D and u relaxes towards U with τ_F (or stays at U if τ_F is 0); at a spike, record u x, deplete x by u x, then raise u by U(1 - u) if τ_F is above 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def tm(times, U, tauD, tauF):
    u, x, last, out = U, 1.0, None, []
    for t in times:
        if last is not None:
            d = t - last
            x = ...
            u = (U + (u - U) * np.exp(-d / tauF)) if tauF > 0 else U
        out.append(u * x)
        x = x - u * x
        if tauF > 0:
            u = ...
        last = t
    return np.array(out)

times = np.arange(10) * 50.0
print("depressing  :", np.round(tm(times, 0.5, 800.0, 0.0), 3))
print("facilitating:", np.round(tm(times, 0.05, 100.0, 1000.0), 3))

In [ ]:
import numpy as np
def test_dep():
    "with no facilitation the model reproduces yesterday's depressing responses"
    a = tm(np.arange(3) * 50.0, 0.5, 800.0, 0.0)
    assert abs(a[0] - 0.5) < 1e-12 and abs(a[1] - 0.5 * (1 - 0.5 * np.exp(-50 / 800))) < 1e-12

def test_fac():
    "the facilitating synapse's response grows over the first ten spikes at 20 Hz"
    a = tm(np.arange(10) * 50.0, 0.05, 100.0, 1000.0)
    assert np.all(np.diff(a) > 0)

test_dep()
test_fac()
print("All checks passed")

### Exercise: Efficacy against rate

For each synapse and each rate, drive the model with 100 regular spikes and store the last response in `eff[name][f]`. Compute the estimate `f_star` (Hz) of the facilitating synapse's preferred rate.

In [ ]:
rates = [1, 2, 5, 10, 15, 20, 40, 80]
params = {"dep": (0.5, 800.0, 0.0), "fac": (0.05, 100.0, 1000.0)}
eff = {name: {} for name in params}
for name, p in params.items():
    for f in rates:
        eff[name][f] = ...
f_star = ...
for f in rates:
    print(f"{f:3d} Hz: depressing {eff['dep'][f]:.3f} per spike ({f * eff['dep'][f]:5.2f}/s), facilitating {eff['fac'][f]:.3f} ({f * eff['fac'][f]:5.2f}/s)")
print(f"estimated preferred rate of the facilitating synapse: {f_star:.1f} Hz")
plt.semilogx(rates, [eff["dep"][f] for f in rates], "o-", label="depressing")
plt.semilogx(rates, [eff["fac"][f] for f in rates], "s-", label="facilitating")
plt.xlabel("rate (Hz)"); plt.ylabel("steady efficacy per spike"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_dep():
    "the depressing synapse's efficacy falls monotonically with rate"
    e = [eff["dep"][f] for f in rates]
    assert all(a > b for a, b in zip(e, e[1:]))

def test_fac():
    "the facilitating synapse's efficacy peaks between 5 and 40 Hz, near the estimate"
    e = [eff["fac"][f] for f in rates]
    peak = rates[int(np.argmax(e))]
    assert 5 <= peak <= 40 and 0.3 < peak / f_star < 3

test_dep()
test_fac()
print("All checks passed")

### Exercise: Recovering three parameters

`amps` holds noisy response amplitudes to an irregular train `spikes` from a synapse with hidden parameters. For each (U, τ_D, τ_F) on the grid, compute the model responses m, choose the best amplitude scale A = (m · amps)/(m · m), and keep the combination with the smallest squared error.

In [ ]:
r = np.random.default_rng(1998)
spikes = np.cumsum(r.exponential(1000 / 15, 40))            # 40 spikes at about 15 Hz, Poisson timing
amps = 2.0 * tm(spikes, 0.1, 200.0, 600.0) * (1 + 0.05 * r.normal(size=40))

best, best_A = (np.inf, None, None, None), None
for U in np.arange(0.02, 0.41, 0.02):
    for tD in (50, 100, 150, 200, 300, 400, 600, 800):
        for tF in (0, 100, 200, 400, 600, 800, 1200):
            m = tm(spikes, U, float(tD), float(tF))
            A = ...
            err = ...
            if err < best[0]:
                best, best_A = (err, U, tD, tF), A
print(f"fit: U = {best[1]:.2f}, tau_D = {best[2]} ms, tau_F = {best[3]} ms, A = {best_A:.2f}  (truth 0.10, 200, 600, 2.0)")
plt.plot(amps, "o", label="data"); plt.plot(best_A * tm(spikes, *best[1:]), label="fit")
plt.xlabel("spike number"); plt.ylabel("response"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the fit recovers U within 0.03, tau_D within 40% and tau_F within 40%"
    U, tD, tF = best[1:]
    assert abs(U - 0.1) < 0.03 and abs(tD - 200) / 200 < 0.4 and abs(tF - 600) / 600 < 0.4

def test_scale():
    "the amplitude scale A is chosen by least squares for each candidate"
    U, tD, tF = best[1:]
    m = tm(spikes, U, tD, tF)
    assert abs(best_A - (m @ amps) / (m @ m)) < 1e-9

test_fit()
test_scale()
print("All checks passed")